In [1]:
import os
from pathlib import Path

In [2]:
import chromadb
from chromadb.config import Settings

In [ ]:
from utils.db_utils import ensure_ltm_vector_collection

In [4]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "output_news" / "test.txt").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root containing output_news/*.json")

In [5]:
PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

In [6]:
CHROMA_STORE_PATH = PROJECT_ROOT / "demo" / "chroma_gemini_handson_2"
PROMOTION_MODEL = "gemini-3.5-flash"
CHATBOT_MODEL = "gemini-3.5-flash"
EMBEDDING_MODEL = "gemini-embedding-001"

In [ ]:
##--------------반드시 입력해주세요!--------------------
GEMINI_API_KEY = ""
##------------------------------------------------------

In [8]:
import json
from google import genai

if not GEMINI_API_KEY:
    raise RuntimeError(".env에 GEMINI_API_KEY 또는 GOOGLE_API_KEY를 설정하세요.")

client = genai.Client(api_key=GEMINI_API_KEY)

In [9]:
ltm_collection = ensure_ltm_vector_collection(chroma_path=CHROMA_STORE_PATH)

[LTM] ChromaDB collection 'ltm_embeddings' ready at C:\Users\chae\project123\mini_o\demo\chroma_gemini_handson_2


In [ ]:
# def compact_embedding(text: str) -> list[float]:
#     response = client.models.embed_content(model=EMBEDDING_MODEL, contents=text)
#     embedding = response.embeddings[0].values
#     return [float(value) for value in embedding]

In [ ]:
from utils.chat_utils import compact_embedding

In [11]:
# 단순 뉴스 기사 관련 질문
chatbot_question1 = "최근 고유가 상황에 대응하는 정부의 정책을 설명해줘"
chatbot_query_embedding1 = compact_embedding(chatbot_question1)

In [12]:
# 뉴스 기사와 주유소 지역별 평균판매가격 분석 질문
chatbot_question2 = "환율과 지역별 주유소 지역별 평균판매가격의 공통된 흐름을 요약해줘."
chatbot_query_embedding2 = compact_embedding(chatbot_question2)

In [13]:
chatbot_ltm_search_results1 = ltm_collection.query(
    query_embeddings=[chatbot_query_embedding1],
    n_results=min(3, max(1, ltm_collection.count())),
    include=["documents", "metadatas", "distances"],
) if ltm_collection.count() else {"ids": [[]], "documents": [[]], "metadatas": [[]], "distances": [[]]}

In [14]:
chatbot_ltm_search_results2 = ltm_collection.query(
    query_embeddings=[chatbot_query_embedding2],
    n_results=min(3, max(1, ltm_collection.count())),
    include=["documents", "metadatas", "distances"],
) if ltm_collection.count() else {"ids": [[]], "documents": [[]], "metadatas": [[]], "distances": [[]]}

In [ ]:
from utils.chat_utils import parse_metadata_json_list, chroma_hits, format_ltm_hit_for_chatbot

In [18]:
chatbot_retrieval_context1 = {
    "query": chatbot_question1,
    "ltm_context": [format_ltm_hit_for_chatbot(hit) for hit in chroma_hits(chatbot_ltm_search_results1)],
}

In [19]:
chatbot_retrieval_context2 = {
    "query": chatbot_question2,
    "ltm_context": [format_ltm_hit_for_chatbot(hit) for hit in chroma_hits(chatbot_ltm_search_results2)],
}

In [ ]:
# chatbot_prompt1 = f"""
# 당신은 학습 메모리를 활용하는 한국어 챗봇입니다.
# 사용자 질문으로 semantic search한 LTM 메모리만 근거로 답하세요.
# 사용자의 이전 질문 패턴을 반영해 사용자 답변에 대답하세요.

# Semantic search 메모리:
# {json.dumps(chatbot_retrieval_context1, ensure_ascii=False, indent=2)}

# 사용자 질문: {chatbot_question1}
# """

In [ ]:
# chatbot_prompt2 = f"""
# 당신은 학습 메모리를 활용하는 한국어 챗봇입니다.
# 사용자 질문으로 semantic search한 LTM 메모리만 근거로 답하세요.
# 사용자의 이전 질문 패턴을 반영해 사용자 답변에 대답하세요.

# Semantic search 메모리:
# {json.dumps(chatbot_retrieval_context2, ensure_ascii=False, indent=2)}

# 사용자 질문: {chatbot_question2}
# """

In [20]:
import pandas as pd

In [21]:
# ✅ CSV는 앱 시작 시 한 번만 로드
df = pd.read_csv("주유소_지역별_평균판매가격.csv", encoding='cp949')
df = df.set_index('구분')
df.index = pd.to_datetime(df.index)  # 날짜 인덱스로 변환

In [22]:
REGIONS = df.columns.tolist()  # ['서울', '부산', ...]

In [23]:
# ✅ Tool 스키마: Gemini가 채울 parameters 명시
select_oil_price_declaration = {
    "name": "select_oil_price",
    "description": (
        "날짜 범위와 지역을 지정해 주유소 일별 평균 판매가격(원/리터)을 조회합니다. "
        "가격 수치가 필요한 질문에만 호출하세요."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "start_date": {
                "type": "string",
                "description": "조회 시작일 (YYYY-MM-DD 형식)",
            },
            "end_date": {
                "type": "string",
                "description": "조회 종료일 (YYYY-MM-DD 형식)",
            },
            "regions": {
                "type": "array",
                "items": {"type": "string"},
                "description": (
                    f"조회할 지역 목록. 생략 시 전체 조회. "
                    f"가능한 값: {REGIONS}"
                ),
            },
        },
        "required": ["start_date", "end_date"],
    },
}

# ✅ 실제 함수: Gemini가 넘긴 인수(start_date, end_date, regions)를 받음
def select_oil_price(start_date: str, end_date: str, regions: list[str] | None = None) -> dict:
    mask = (df.index >= start_date) & (df.index <= end_date)
    result = df.loc[mask]

    if regions:
        result = result[regions]

    return {
        str(idx.date()): row.dropna().to_dict()
        for idx, row in result.iterrows()
    }

In [24]:
from google.genai import types

In [25]:
system_instruction = """
당신은 학습 메모리를 활용하는 한국어 챗봇입니다.
사용자 질문으로 semantic search한 LTM 메모리를 근거로 답하세요.
사용자의 이전 질문 패턴을 반영해 답변하세요.
주유소 가격 등 수치 데이터가 필요한 경우에만 select_oil_price 툴을 호출하세요.
툴 없이 답할 수 있으면 호출하지 마세요.
"""

In [26]:
# Configure the client and tools
# client = genai.Client()
tools = types.Tool(function_declarations=[select_oil_price_declaration])
config = types.GenerateContentConfig(
    tools=[tools],
    system_instruction=system_instruction,
)

In [27]:
# ✅ Tool 호출 처리 루프 포함한 질의 함수
def ask(question: str, retrieval_context: dict) -> str:
    # LTM 메모리는 contents(user 메시지) 안에 포함
    user_prompt = f"""
Semantic search 메모리:
{json.dumps(retrieval_context, ensure_ascii=False, indent=2)}

사용자 질문: {question}
"""
    contents = [
        types.Content(role="user", parts=[types.Part(text=user_prompt)])
    ]

    # Tool 호출 루프
    while True:
        response = client.models.generate_content(
            model=CHATBOT_MODEL,
            contents=contents,
            config=config,
        )

        candidate = response.candidates[0].content
        contents.append(candidate)

        tool_calls = [p for p in candidate.parts if p.function_call]
        if not tool_calls:
            return response.text  # 최종 답변

        # Tool 실행
        tool_results = []
        for part in tool_calls:
            fc = part.function_call
            args = dict(fc.args)

            if fc.name == "select_oil_price":
                result = select_oil_price(**args)
            else:
                result = {"error": f"알 수 없는 함수: {fc.name}"}

            tool_results.append(
                types.Part(
                    function_response=types.FunctionResponse(
                        name=fc.name,
                        response={"result": result},
                    )
                )
            )

        contents.append(
            types.Content(role="tool", parts=tool_results)
        )

In [28]:
answer1 = ask(chatbot_question1, chatbot_retrieval_context1)

In [29]:
answer2 = ask(chatbot_question2, chatbot_retrieval_context2)

In [30]:
answer1

'최근 국제 유가 급등 및 중동발 유가 충격에 대응하여, 정부와 지자체는 민생 부담을 완화하고 물가를 안정시키기 위해 다음과 같은 정책적 노력을 시행하고 있습니다.\n\n1. **유류세 인하 조치 연장**\n   * 유가 충격을 최소화하고 소비자 물가 급등세를 방어하기 위해, 종료 예정이던 **유류세 인하 조치를 2개월 연장**하여 적용하고 있습니다.\n\n2. **석유 최고가격제 적용 및 동결**\n   * 고유가 상황이 시장에 미치는 충격을 완화하고 물가를 억제하기 위해 **석유 최고가격제를 도입하고 이를 동결**하는 조치를 단행하고 있습니다.\n\n3. **지자체 고유가 지원금 지급**\n   * 정부 차원의 대책과 더불어, 각 지방자치단체에서도 주민들의 민생 부담을 덜어주기 위해 **고유가 지원금**을 지급하는 등 다각적인 지원 정책을 펼치고 있습니다.\n\n이처럼 정부는 유류세 인하와 가격 통제, 직접적인 지원금 지급 등을 통해 고유가로 인한 국민들의 경제적 부담을 줄이고 물가 안정을 도모하고 있습니다.'

In [31]:
answer2

'제공된 메모리를 바탕으로 **환율**과 **국내 주유소 판매가격(유가)**의 공통된 흐름과 상호 관계를 요약해 드립니다.\n\n---\n\n### 1. 대외 지정학적 리스크에 따른 동반 상승 (고환율·고유가 기조)\n* **공통 원인:** 중동 전쟁 장기화 및 중동 정세 불안 등 지정학적 리스크가 환율과 유가 상승을 동시에 자극하고 있습니다.\n* **흐름:** 원·달러 환율이 1,500원대를 상회하는 고환율 기조를 보이는 동시에, 국제 유가 역시 요동치며 상승 압력을 받고 있습니다. 달러화 강세(고환율)와 원유 가격 상승이 동시에 발생하며 국내 경제에 부담을 주는 흐름입니다.\n\n### 2. 시차를 둔 국내 주유소 가격 반영\n* **흐름:** 국제 유가와 환율의 변동은 보통 **2~3주의 시차**를 두고 국내 지역별 주유소의 휘발유 및 경유 평균 판매가격에 반영됩니다. 국제 제품 가격이 하락세로 돌아서면 국내 주유소 가격도 시차를 두고 하락 흐름을 타게 됩니다.\n\n### 3. 국내 소비자물가 및 실물 경제 압박\n* **공통 영향:** 고환율로 인해 원화 가치가 떨어지면 원유 수입 비용이 추가로 상승하게 되며, 이는 국내 주유소 기름값 상승으로 직결됩니다. \n* 결과적으로 고환율과 고유가가 맞물려 **국내 소비자물가 상승 압력**을 가중시키고, 지역 경제의 생산·소비 둔화 및 금리 인상 우려 등 경기 전반의 불확실성을 높이는 방향으로 흘러가고 있습니다.'

In [34]:
from IPython.display import Markdown, display

In [36]:
display(Markdown(answer1))

최근 국제 유가 급등 및 중동발 유가 충격에 대응하여, 정부와 지자체는 민생 부담을 완화하고 물가를 안정시키기 위해 다음과 같은 정책적 노력을 시행하고 있습니다.

1. **유류세 인하 조치 연장**
   * 유가 충격을 최소화하고 소비자 물가 급등세를 방어하기 위해, 종료 예정이던 **유류세 인하 조치를 2개월 연장**하여 적용하고 있습니다.

2. **석유 최고가격제 적용 및 동결**
   * 고유가 상황이 시장에 미치는 충격을 완화하고 물가를 억제하기 위해 **석유 최고가격제를 도입하고 이를 동결**하는 조치를 단행하고 있습니다.

3. **지자체 고유가 지원금 지급**
   * 정부 차원의 대책과 더불어, 각 지방자치단체에서도 주민들의 민생 부담을 덜어주기 위해 **고유가 지원금**을 지급하는 등 다각적인 지원 정책을 펼치고 있습니다.

이처럼 정부는 유류세 인하와 가격 통제, 직접적인 지원금 지급 등을 통해 고유가로 인한 국민들의 경제적 부담을 줄이고 물가 안정을 도모하고 있습니다.

In [37]:
display(Markdown(answer2))

제공된 메모리를 바탕으로 **환율**과 **국내 주유소 판매가격(유가)**의 공통된 흐름과 상호 관계를 요약해 드립니다.

---

### 1. 대외 지정학적 리스크에 따른 동반 상승 (고환율·고유가 기조)
* **공통 원인:** 중동 전쟁 장기화 및 중동 정세 불안 등 지정학적 리스크가 환율과 유가 상승을 동시에 자극하고 있습니다.
* **흐름:** 원·달러 환율이 1,500원대를 상회하는 고환율 기조를 보이는 동시에, 국제 유가 역시 요동치며 상승 압력을 받고 있습니다. 달러화 강세(고환율)와 원유 가격 상승이 동시에 발생하며 국내 경제에 부담을 주는 흐름입니다.

### 2. 시차를 둔 국내 주유소 가격 반영
* **흐름:** 국제 유가와 환율의 변동은 보통 **2~3주의 시차**를 두고 국내 지역별 주유소의 휘발유 및 경유 평균 판매가격에 반영됩니다. 국제 제품 가격이 하락세로 돌아서면 국내 주유소 가격도 시차를 두고 하락 흐름을 타게 됩니다.

### 3. 국내 소비자물가 및 실물 경제 압박
* **공통 영향:** 고환율로 인해 원화 가치가 떨어지면 원유 수입 비용이 추가로 상승하게 되며, 이는 국내 주유소 기름값 상승으로 직결됩니다. 
* 결과적으로 고환율과 고유가가 맞물려 **국내 소비자물가 상승 압력**을 가중시키고, 지역 경제의 생산·소비 둔화 및 금리 인상 우려 등 경기 전반의 불확실성을 높이는 방향으로 흘러가고 있습니다.